In [1]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
apikey = os.getenv("OPENAI_API_KEY")
default_model = os.getenv("OPENAI_DEFAULT_MODEL")
g_apikey = os.getenv("GEMINI_API_KEY")
g_default_model = os.getenv("GEMINI_DEFAULT_MODEL")

if not apikey:
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to the .env file.")
if not default_model:
    raise RuntimeError("OPENAI_DEFAULT_MODEL is not set. Add it to the .env file.")

default_model = default_model.strip().lower()
print(f"Using model: {default_model}")
print(f"Gemini API key configured: {bool(g_apikey)}")
print(f"Gemini model: {g_default_model}")

llm = init_chat_model('openai:gpt-6-luna')

Using model: gpt-6-luna
Gemini API key configured: True
Gemini model: gemini-3.5-flash-lite


In [12]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

questions = [
    "부산 여행에서 딱 한 곳만 가봐야 한다면 어떤 곳인지 알려주세요",
    "부산역에서 그곳에 가는 교통편을 알려주세요",
    "그 근처에서 먹을만한 음식점들을 추천해주세요",
]

messages = [
    ('system', '당신은 여행 전문가로 고객의 여행 일정에 도움을 줍니다')
]

# 전문가 역할: 처음 한 번만 출력
print(type(messages[0]), messages[0])

for question in questions:
    messages.append(('user', question))

    response = llm.invoke(messages)
    messages.append(('assistant', response.content))

    # 새 질문과 답변만 순서대로 출력
    print(type(messages[-2]), messages[-2])
    print(type(response), response)


<class 'tuple'> ('system', '당신은 여행 전문가로 고객의 여행 일정에 도움을 줍니다')
<class 'tuple'> ('user', '부산 여행에서 딱 한 곳만 가봐야 한다면 어떤 곳인지 알려주세요')
<class 'langchain_core.messages.ai.AIMessage'> content='딱 한 곳이라면 **광안리해수욕장**을 추천해요. 바다 너머로 **광안대교**가 보여 낮과 밤의 분위기가 모두 좋고, 특히 해가 진 뒤 조명이 켜질 때 부산다운 풍경을 즐길 수 있어요. 해변을 따라 카페와 식당도 많아 산책과 식사를 한곳에서 해결하기 편합니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 326, 'prompt_tokens': 45, 'total_tokens': 371, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 227, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EYhEQKnZbeCcvqN2z73jiaW459XwZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a137d4-92b2-7e00-8db

In [ ]:
# 수업자 파일
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=default_model)

messages = [
    ("system", "당신은 여행 전문가로 고객의 여행 일정에 도움을 줍니다."),
]

questions = [
    "부산 여행에서 딱 한 곳만 가 봐야 한다면 어떤 곳인지 알려 주세요.",
    "부산역에서 그곳에 가는 교통편을 알려 주세요.",
    "그 근처에서 먹을 만한 음식점들을 추천해 주세요.",
]

for i, question in enumerate(questions, start=1):
    messages.append(("human", question))

    response = llm.invoke(messages)
    answer = response.content.strip()
    messages.append(("ai", answer))

    usage = response.usage_metadata or {}

    print(f"\n[Q{i}] {question}")
    print(f"[A{i}] {answer}")
    print(f"[토큰] 입력 {usage.get('input_tokens')} / 출력 {usage.get('output_tokens')} / 합계 {usage.get('total_tokens')}")

print("\n누적 메시지 수:", len(messages))

In [15]:
# LangChain의 대화 메시지 기록 관리 클래스 임포트
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import SystemMessage, HumanMessage

memory = InMemoryChatMessageHistory() 

memory.add_message(('system','당신은 여행 전문가로 고객의 여행 일정에 도움을 줍니다'))
memory.add_user_message('부산 여행에서 딱 한 곳만을 가봐야 한다면 어떤 곳인지 알려주세요.')

response = llm.invoke(memory.messages)

memory.add_ai_message(response.content)

memory.add_user_message("부산역에서 그곳에 가는 교통편을 알려주세요.")
response = llm.invoke(memory.messages)
memory.add_ai_message(response.content)


for msg in memory.messages:
    print(type(msg), msg)

<class 'tuple'> ('system', '당신은 여행 전문가로 고객의 여행 일정에 도움을 줍니다')
<class 'langchain_core.messages.human.HumanMessage'> content='부산 여행에서 딱 한 곳만을 가봐야 한다면 어떤 곳인지 알려주세요.' additional_kwargs={} response_metadata={}
<class 'langchain_core.messages.ai.AIMessage'> content='딱 한 곳만 간다면 **광안리 해변**을 추천해요. 부산 바다와 광안대교를 한눈에 볼 수 있고, 해 질 무렵부터 밤까지 풍경이 특히 아름답습니다. 해변을 따라 걷거나 바다를 바라보며 식사하기도 좋아서 부산의 분위기를 제대로 느낄 수 있어요.' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
<class 'langchain_core.messages.human.HumanMessage'> content='부산역에서 그곳에 가는 교통편을 알려주세요.' additional_kwargs={} response_metadata={}
<class 'langchain_core.messages.ai.AIMessage'> content='부산역에서 **광안리 해변**까지는 지하철이 편리합니다.\n\n- 부산역에서 **지하철 1호선**을 타고 **서면역**으로 이동합니다.\n- 서면역에서 **2호선 장산 방면**으로 갈아타고 **광안역**에서 내립니다.\n- 광안역에서 해변까지는 출구에서 도보로 약 **10~15분**입니다.\n\n전체 이동은 대략 **40~50분** 정도 걸려요. 짐이 많거나 빠르게 가고 싶다면 택시로 약 **25~35분** 정도지만, 교통 상황에 따라 달라질 수 있습니다.' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]


In [11]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

chatbot = create_agent(
    model = llm, 
    tools=[], 
    system_prompt = '당신은 친절한 챗봇입니다' ,
    checkpointer = InMemorySaver() 
)

cfg = {'configurable': {'thread_id': '1'}}

for chat in ['안녕하세요 ', '나는 홍길동입니다', '내 이름이 뭐에요?']:
    response = chatbot.invoke({'messages':[('user',chat)]}, config = cfg)
    print("-" * 30)
    print(f"질문: {chat}")
    print(f"답변: {response['messages'][-1].text}\n")

------------------------------
질문: 안녕하세요 
답변: 안녕하세요! 무엇을 도와드릴까요?

------------------------------
질문: 나는 홍길동입니다
답변: 반가워요, 홍길동님! 😊

------------------------------
질문: 내 이름이 뭐에요?
답변: 홍길동님이에요.



In [5]:
# 두 번째 사람과의 대화라면
cfg = {'configurable': {'thread_id': '2'}}

chat = '내 이름이 뭐였죠?'
response = chatbot.invoke({'messages':[('user',chat)]}, config = cfg)
print('USER', chat)
print("Bot", response['messages'][-1].text)
print("-" * 30)

USER 내 이름이 뭐였죠?
Bot 아직 이름을 알려주시지 않았어요. 어떤 이름으로 불러드릴까요?
------------------------------


2-1. 한계점
user1= [
    "나는 일본 여행을 준비 중이야.",
    "오사카와 교토를 방문할 예정이야.",
    "내가 어디를 여행하려고 했지?",
]
user2= [
    "우리 백화점 이름은 행복백화점이야.",
    "이번 달 할인 행사는 여름 정기세일이야.",
    "우리 백화점 이름이 뭐였지?",
]
user3= [
    "우리 식당 이름은 맛있는집이야.",
    "대표 메뉴는 김치찌개야.",
    "우리 대표 메뉴가 뭐였지?",
]

In [13]:
state = chatbot.get_state(cfg)
for msg in state.values['messages']:
    print(msg.type, msg.text)

human 안녕하세요 
ai 안녕하세요! 무엇을 도와드릴까요?
human 나는 홍길동입니다
ai 반가워요, 홍길동님! 😊
human 내 이름이 뭐에요?
ai 홍길동님이에요.


In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.chat_models import init_chat_model
import sqlite3

llm = init_chat_model('openai:gpt-6-luna')

conn = sqlite3.connect('chatbot.db', check_same_thread = False)
saver = SqliteSaver(conn)

chatbot = create_agent(
    model=llm,
    tools=[],
    system_prompt = '너는 친절한 챗봇이야',
    checkpointer = saver
)
# cfg = {'configurable':{'thread_id': '1'}}

# for chat in ['안녕하세요', '나는 홍길동입니다', '내 이름이 뭐에요?']:
#     response = chatbot.invoke({'messages':[('user',chat)]}, config=cfg)
#     print('USER', chat)
#     print("Bot", response['messages'][-1].text)
#     print('-'*30)

In [ ]:
# 2차 실행 (재시작 후에도 이전 대화 기억) 
cfg = {'configurable':{'thread_id': '1'}}

chat = '제 이름이 뭐였죠?'
result = chatbot.invoke({"messages": [("user", chat)]}, 
                    config=cfg )
print(f"Q: {question}\nA: {result['messages'][-1].text}\n")

Q: 제 이름이 뭐였죠? 제게 맞는 서울 음식점 한 곳 추천해 주세요.
A: 홍길동님이세요.



In [11]:
from langchain_core.messages import trim_messages

# WINDOW_K = 3   # 최근 3턴(질문+답변)만 기억  5로 수정해보기
WINDOW_K = 5   # 최근 3턴(질문+답변)만 기억  5로 수정해보기

def window(messages, k=WINDOW_K):
    return trim_messages(
        messages,
        strategy="last",
        token_counter=len,
        max_tokens=k * 2 + 2,   # system 1개 + 최근 k턴(2개씩) + 새 질문 1개
        include_system=True,
        start_on="human",
    )

history = [("system", "당신은 서울 여행 가이드입니다. 짧게 답하세요.")]
questions = [
    "서울을 관광하고 있습니다. 서울 음식점 한 곳만 알려 주세요.",      # 1턴: 맥락 생성
    "그곳의 제일 인기 있는 메뉴는 무엇인가요?",                      # 2턴: 윈도우 안 → 기억함
    "10월 서울 날씨에는 옷을 어떻게 입는 게 좋을까요?",               # 3턴: 무관한 질문
    "경복궁 관람 시간은 어떻게 되나요?",                             # 4턴: 무관한 질문
    "저녁에 산책하기 좋은 한강공원 한 곳만 추천해 주세요.",            # 5턴: 무관한 질문
    "처음에 알려 준 음식점까지 서울역에서 대중교통으로 어떻게 가나요?",  # 6턴: 1턴이 잘려 → 잊음
]

for q in questions:
    history.append(("human", q))
    response = llm.invoke(window(history))   # 최근 k턴만 전달
    history.append(("ai", response.content))
    print(f"Q: {q}\nA: {response.content}\n")

print("전체 기록 메시지 수 :", len(history))
print("LLM에 전달되는 메시지 수:", len(window(history + [("human", "다음 질문")])))

Q: 서울을 관광하고 있습니다. 서울 음식점 한 곳만 알려 주세요.
A: 명동교자 본점에서 칼국수와 만두를 드셔 보세요. 명동역 근처라 관광 중 들르기 편합니다.

Q: 그곳의 제일 인기 있는 메뉴는 무엇인가요?
A: 명동교자의 대표 인기 메뉴는 **칼국수**입니다. 마늘 향이 강한 김치와 함께 드셔 보세요.

Q: 10월 서울 날씨에는 옷을 어떻게 입는 게 좋을까요?
A: 10월 서울은 낮엔 선선하고 아침저녁엔 쌀쌀한 편이라 **긴팔에 가벼운 재킷이나 트렌치코트**를 겹쳐 입는 게 좋아요. 월말에는 더 두툼한 외투가 필요할 수 있고, 작은 우산도 챙기세요.

Q: 경복궁 관람 시간은 어떻게 되나요?
A: 10월 경복궁은 **오전 9시부터 오후 6시까지** 관람할 수 있으며, 입장은 **오후 5시까지**입니다. **매주 화요일은 휴궁**이에요.

Q: 저녁에 산책하기 좋은 한강공원 한 곳만 추천해 주세요.
A: **여의도 한강공원**을 추천해요. 강변 산책로가 잘 정비돼 있고, 저녁에는 한강과 도심 야경을 함께 즐기기 좋아요.

Q: 처음에 알려 준 음식점까지 서울역에서 대중교통으로 어떻게 가나요?
A: 서울역에서 **지하철 4호선**을 타고 명동역에서 내리세요(한 정거장). **8번 출구**로 나와 명동교자 본점까지 걸으면 약 5분입니다.

전체 기록 메시지 수 : 13
LLM에 전달되는 메시지 수: 12


In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

summary_bot = create_agent(
    model=llm,
    tools=[],
    system_prompt="당신은 서울 여행 가이드입니다. 짧게 답하세요.",
    middleware=[
        SummarizationMiddleware(model=llm, trigger=("messages", 6), keep=("messages", 2)),
    ],
    checkpointer=InMemorySaver(),
)

cfg = {"configurable": {"thread_id": "summary-1"}}
questions = [
    "서울을 관광하고 있습니다. 서울 음식점 한 곳만 알려 주세요.",      # 1턴: 맥락 생성
    "그곳의 제일 인기 있는 메뉴는 무엇인가요?",                      # 2턴: 윈도우 안 → 기억함
    "10월 서울 날씨에는 옷을 어떻게 입는 게 좋을까요?",               # 3턴: 무관한 질문
    "경복궁 관람 시간은 어떻게 되나요?",                             # 4턴: 무관한 질문
    "저녁에 산책하기 좋은 한강공원 한 곳만 추천해 주세요.",            # 5턴: 무관한 질문
    "처음에 알려 준 음식점까지 서울역에서 대중교통으로 어떻게 가나요?",  # 6턴: 1턴이 잘려 → 잊음
]
# 저장된 기록 확인: 앞부분이 요약 메시지로 바뀌어 있습니다.
for q in questions :
    result = summary_bot.invoke({'messages':[("human", q)]}, config=cfg)
    print(f"Q: {q}\n A: {result['messages'][-1].text}\n")
    
    

Q: 서울을 관광하고 있습니다. 서울 음식점 한 곳만 알려 주세요.
 A: 명동교자 본점에서 칼국수와 만두를 드셔 보세요. 서울의 대표적인 칼국수 맛집입니다.

Q: 그곳의 제일 인기 있는 메뉴는 무엇인가요?
 A: 대표 메뉴는 **칼국수**예요. 진한 국물과 만두가 들어간 것으로 유명합니다.

Q: 10월 서울 날씨에는 옷을 어떻게 입는 게 좋을까요?
 A: 10월 서울은 대체로 선선해요. 긴팔 옷에 가벼운 재킷이나 가디건을 겹쳐 입고, 아침저녁엔 얇은 코트가 유용합니다. 비에 대비해 작은 우산도 챙기세요.

Q: 경복궁 관람 시간은 어떻게 되나요?
 A: 10월 경복궁은 **오전 9시~오후 6시** 관람할 수 있고, **입장 마감은 오후 5시**예요. **매주 화요일은 휴궁**입니다. ※ 운영 시간은 변경될 수 있으니 방문 전 공식 안내를 확인하세요.

Q: 저녁에 산책하기 좋은 한강공원 한 곳만 추천해 주세요.
 A: **반포 한강공원**을 추천해요. 강변 산책로가 잘 정비되어 있고, 저녁에는 한강과 반포대교의 야경을 즐기기 좋아요.

Q: 처음에 알려 준 음식점까지 서울역에서 대중교통으로 어떻게 가나요?
 A: 서울역에서 **지하철 4호선 당고개 방면**을 타고 **명동역에서 한 정거장** 가세요. 8번 출구로 나와 명동교자 본점까지 걸어서 약 5분입니다.



# 캐싱 시작

In [16]:
from langchain_core.globals import set_llm_cache
from langchain_community.cache import InMemoryCache
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate

# 메모리 기반 캐시 활성화
set_llm_cache(InMemoryCache())

prompt = ChatPromptTemplate.from_template(
  "서울 남대문시장에서 가장 맛있는 길거리 음식은 무엇인가요?")

chain = prompt | llm

C:\Users\KDA\AppData\Local\Temp\ipykernel_25124\1798957936.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.cache import InMemoryCache


In [19]:
from langchain_core.globals import set_llm_cache
from langchain_community.cache import InMemoryCache
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate

# 메모리 기반 캐시 활성화
set_llm_cache(InMemoryCache())

prompt = ChatPromptTemplate.from_template("서울 맛집 3곳 알려줘")

chain = prompt | llm

In [20]:
%%time
response = chain.invoke({})
print(response.content)

서울에서 메뉴별로 세 곳 추천할게요.

1. **명동교자 본점** — 명동  
   진한 국물의 칼국수와 만두가 유명해요. 식사 시간엔 대기가 있을 수 있어요.

2. **금돼지식당** — 약수  
   두툼한 돼지고기를 구워 먹는 곳이에요. 저녁엔 웨이팅이 잦습니다.

3. **우래옥** — 을지로  
   평양냉면과 불고기로 잘 알려져 있어요. 가격대는 비교적 높은 편이에요.

영업시간이나 대기 상황은 방문 전에 확인해 보세요.
CPU times: total: 15.6 ms
Wall time: 6.19 s


In [21]:
%%time
response = chain.invoke({})
print(response.content)

서울에서 메뉴별로 세 곳 추천할게요.

1. **명동교자 본점** — 명동  
   진한 국물의 칼국수와 만두가 유명해요. 식사 시간엔 대기가 있을 수 있어요.

2. **금돼지식당** — 약수  
   두툼한 돼지고기를 구워 먹는 곳이에요. 저녁엔 웨이팅이 잦습니다.

3. **우래옥** — 을지로  
   평양냉면과 불고기로 잘 알려져 있어요. 가격대는 비교적 높은 편이에요.

영업시간이나 대기 상황은 방문 전에 확인해 보세요.
CPU times: total: 15.6 ms
Wall time: 2.53 ms


In [ ]:
yes_llm = init_chat_model('openai:gpt-6-luna')
no_llm = init_chat_model('openai:gpt-6-luna', cache = False)
question = ChatPromptTemplate.from_template("서울에서 맛있는 음식점 3곳 알려줘")

In [25]:
%%time
yes_llm.invoke(question)

CPU times: total: 0 ns
Wall time: 0 ns


AIMessage(content='성함은 아직 알려주지 않으셔서 모르겠어요. 취향도 아직 모르니, 무난한 한 곳으로 **광화문 미진**을 추천할게요. 메밀국수로 유명하고, 광화문 근처에서 간단히 한 끼 먹기 좋아요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 429, 'prompt_tokens': 26, 'total_tokens': 455, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 354, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EYj6uqKIsypLYOQwkPVcg8DIoHgHn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a13842-c7a8-7c53-84f1-cf9a5bc78e3a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 429, 'total_tokens': 455, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creatio

In [24]:
%%time
no_llm.invoke(question)

CPU times: total: 15.6 ms
Wall time: 4.84 s


AIMessage(content='아직 이름을 알려주지 않으셔서 제가 알 수 없어요. 취향도 아직 모르니 무난한 곳으로 한 군데 추천드리면, **명동교자 명동 본점**은 어떠세요? 칼국수와 만두가 유명하고 혼자 식사하기도 편한 편이에요. 붐빌 수 있으니 방문 전에 영업시간을 확인해 보세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 266, 'prompt_tokens': 26, 'total_tokens': 292, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 167, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EYj75or7QoigcdNiaUGb9xOpsOEas', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a13842-f570-76d3-92b6-54b5dc8d5e45-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 266, 'total_tokens': 292, 'input_token_details': {'

In [28]:
from urllib.parse import quote_plus
from dotenv import load_dotenv

load_dotenv(override=True)
host = os.getenv("REDIS_HOST").strip()
port = int(os.getenv("REDIS_PORT", 6379))
password = os.getenv("REDIS_PASSWORD").strip()

redis_url = f"redis://default:{quote_plus(password)}@{host}:{port}"

In [29]:
import redis
from langchain_redis import RedisChatMessageHistory

# Redis 클라이언트: 연결 확인 및 직접 조회용
redis_client = redis.Redis.from_url(redis_url, protocol=2)   # decode_responses는 지정하지 않음

history = RedisChatMessageHistory(
    session_id="user-1",
    redis_client=redis_client,
    ttl=3600, # 선택: 1시간 뒤 자동 삭제 (초 단위)
)
print("Redis 연결:", redis_client.ping())

Redis 연결: True


C:\Users\KDA\AppData\Local\Temp\ipykernel_25124\1718100443.py:7: LangChainDeprecationWarning: The class `RedisChatMessageHistory` was deprecated in LangChain 0.2.6 and will be removed in 1.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  history = RedisChatMessageHistory(


In [ ]:
from langchain_core.globals import set_llm_cache
from langchain_redis import RedisCache
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=default_model)

set_llm_cache(RedisCache(redis_client=redis_client))   # Redis 캐시 활성화

prompt = ChatPromptTemplate.from_template("서울 남대문시장에서 가장 맛있는 길거리 음식 5개를 소개해 주세요.")
chain = prompt | llm

In [33]:
%%time

response = chain.invoke({})
print(response)

content='동대문 일대에서 즐기기 좋은 길거리 음식 5가지를 꼽아 보면요. 입맛에 따라 다르지만, 처음 방문한다면 아래 메뉴들이 무난한 선택입니다.\n\n1. **녹두빈대떡** — 겉은 바삭하고 속은 고소해요. 동대문에서 가까운 **광장시장**에서 특히 유명합니다.\n2. **마약김밥** — 한입 크기 김밥에 겨자소스를 곁들여 먹는 별미예요. 역시 광장시장 대표 메뉴입니다.\n3. **떡볶이와 튀김** — 매콤한 떡볶이 국물에 바삭한 튀김을 찍어 먹으면 든든한 간식이 됩니다.\n4. **어묵** — 시장을 둘러보다 따뜻한 국물과 함께 간단히 먹기 좋아요.\n5. **호떡** — 달콤한 속과 노릇한 반죽이 매력적인 간식으로, 추운 날 특히 잘 어울립니다.\n\n참고로 광장시장은 동대문 상권과 가까운 별도 시장이에요. **빈대떡과 마약김밥을 맛보고 싶다면 광장시장 쪽**, 다른 간식은 동대문 일대 노점과 분식집을 함께 둘러보면 좋습니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 795, 'prompt_tokens': 25, 'total_tokens': 820, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 482, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id

In [34]:
from typing import Any, Optional
from langchain_core.caches import BaseCache, RETURN_VAL_TYPE
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document


class SimpleSemanticCache(BaseCache):
    def __init__(self, embedding, similarity_threshold: float = 0.8):
        self.embedding = embedding
        self.similarity_threshold = similarity_threshold
        self.vectorstore = InMemoryVectorStore(embedding)
        self.store: dict[str, RETURN_VAL_TYPE] = {}

    def lookup(self, prompt: str, llm_string: str) -> Optional[RETURN_VAL_TYPE]:
        if not self.store:
            return None
        doc, score = self.vectorstore.similarity_search_with_score(prompt, k=1)[0]
        print(f"  (가장 비슷한 질문 유사도: {score:.3f})")
        if score >= self.similarity_threshold and doc.metadata["llm_string"] == llm_string:
            return self.store[doc.metadata["key"]]
        return None

    def update(self, prompt: str, llm_string: str, return_val: RETURN_VAL_TYPE) -> None:
        key = f"{prompt}::{llm_string}"
        self.store[key] = return_val
        self.vectorstore.add_documents(
            [Document(page_content=prompt, metadata={"llm_string": llm_string, "key": key})]
        )

    def clear(self, **kwargs: Any) -> None:
        self.vectorstore = InMemoryVectorStore(self.embedding)
        self.store = {}

In [35]:
import time
from langchain_core.globals import set_llm_cache
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
set_llm_cache(SimpleSemanticCache(embedding=embeddings, similarity_threshold=0.8))

for q in ["대한민국의 수도는 어디인가요?", "한국 수도 어디야"]:
    start = time.time()
    response = llm.invoke(q)
    print(f"질문: {q}")
    print(f"답변: {response.content}")
    print(f"소요 시간: {time.time() - start:.2f}초\n")

질문: 대한민국의 수도는 어디인가요?
답변: 대한민국의 수도는 서울특별시입니다.
소요 시간: 5.72초

  (가장 비슷한 질문 유사도: 0.992)
질문: 한국 수도 어디야
답변: 대한민국의 수도는 서울특별시입니다.
소요 시간: 0.15초



In [36]:
import time
from langchain_core.globals import set_llm_cache
from langchain_openai import OpenAIEmbeddings
from langchain_redis import RedisSemanticCache

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

semantic_cache = RedisSemanticCache(
    embeddings=embeddings,
    redis_url=redis_url,        # 앞에서 만든 Redis 접속 URL 재사용
    distance_threshold=0.2,     # 거리 기준: 작을수록 엄격 (유사도 0.8 ≈ 거리 0.2)
    ttl=3600,                   # 선택: 캐시 1시간 뒤 자동 삭제
)
semantic_cache.clear()          # 실습용: 이전 실행에서 남은 캐시 제거

set_llm_cache(semantic_cache)

for q in ["대한민국의 수도는 어디인가요?", "한국 수도 어디야"]:
    start = time.time()
    response = llm.invoke(q)
    print(f"질문: {q}")
    print(f"답변: {response.content}")
    print(f"소요 시간: {time.time() - start:.2f}초\n")

질문: 대한민국의 수도는 어디인가요?
답변: 대한민국의 수도는 서울특별시입니다.
소요 시간: 2.22초

질문: 한국 수도 어디야
답변: 대한민국의 수도는 서울특별시입니다.
소요 시간: 0.17초



In [ ]:
# 람다복습
def kkkk(a,b):
    return a + b
ret = kkkk(5,6)
print(ret)


add = lambda a, b: a + b
print(add(5, 6))

11
11


In [ ]:
func_a x+1 = 결과값
func_b 결과값1 * 2 = 결과값2

chain = RunnableLambda(func_a) | RunnableLambda(func_b) | prompt | llm | outputparser

In [ ]:
from langchain_core.globals import set_llm_cache
set_llm_cache(None)

In [ ]:
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser


def func_a(x):
    return x + 1

def func_b(x):
    return x * 2

chain1 = (RunnableLambda(func_a) 
        | RunnableLambda(func_b)
        | RunnableLambda(lambda x : f'숫자{x}에 대해 재미 있는 이야기 알려줘' )
        | llm
        | StrOutputParser())
print(chain1.invoke(3))

### 숫자 8의 모험

어느 날 숫자 8은 거울 앞에 섰어요.

“어라? 위아래가 똑같이 생겼네! 내가 나를 두 번 만났잖아?”

신이 난 8은 빙글빙글 돌며 친구들에게 자랑했어요. 숫자 3은 “난 반으로 접으면 안 똑같은데!” 하고 부러워했고, 숫자 0은 “나도 동그랗긴 한데, 난 가운데가 비었어”라며 씩 웃었죠.

그때 수학 나라의 문이 열리며 옆으로 누운 8이 나타났어요.

“안녕, 나는 무한대야! 끝없이 계속되는 일을 맡고 있지.”

8은 깜짝 놀랐어요. “그럼 내가 옆으로 누우면 끝없이 달릴 수 있는 거야?”

“그렇지!”

8은 옆으로 누워 긴 여행을 떠났어요. 한 바퀴, 두 바퀴, 세 바퀴… 아무리 달려도 끝이 없었죠.

그런데 잠시 뒤, 8은 다시 원래 모습으로 돌아왔어요.

“역시 나는 똑바로 서 있을 때가 제일 편해. 끝없는 여행은 다음에 또 하지 뭐!”

그날부터 8은 똑바로 서 있을 땐 **재미있는 숫자**, 옆으로 누울 땐 **끝없는 모험가**가 되었답니다.


In [ ]:
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser


chain1 = (
    RunnableLambda(
        lambda name: (
            f"{name}에 어울리는 별명을 하나 만들어줘. "
            f"'{name}에 대한 별명: 별명 이모티콘' 형식으로 한 줄만 답하고, "
            "이모티콘은 정확히 하나만 사용해."
        )
    )
    | llm
    | StrOutputParser()
)

print(chain1.invoke('홍길동'))

홍길동에 대한 별명: 동에 번쩍 서에 번쩍 홍대장 🦸


In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model('openai:gpt-6-luna')

first = RunnableLambda(lambda x : f'{x} 이름에 어울리는 별명 하나만 만들어 줘, 별명만 응닿배줘') 
last = RunnableLambda(lambda x: f'{x.content} ★')
print(chain1.invoke('홍길동'))

In [ ]:
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

# 입력된 질문에 오늘 날짜를 더해 프롬프트가 사용할 값으로 전달
def add_today(inputs):
    today = datetime.now()
    return {
        **inputs,
        "date": f"{today.year}년 {today.month}월 {today.day}일",
    }

date_runnable = RunnableLambda(add_today)

# 점심 질문에 답할 때 날짜와 네 가지 음식 종류를 모두 포함하도록 형식을 지정
prompt = ChatPromptTemplate.from_template(
    "아래 형식을 지켜서 답변해 주세요. 첫 줄은 반드시 \"{date}의 점심 메뉴를 추천해드릴게요!\"로 시작하세요.\n\n"
    "질문: {question}\n\n"
    "각 종류의 메뉴를 하나씩 추천하고, 목록 뒤에는 짧은 코멘트를 한 줄 추가해 주세요.\n"
    "1. **한식**: 메뉴 이름\n"
    "2. **중식**: 메뉴 이름\n"
    "3. **일식**: 메뉴 이름\n"
    "4. **양식**: 메뉴 이름\n\n"
    "오늘의 추천을 마무리하는 코멘트 한 줄" 
)

chain = date_runnable | prompt | llm

# 오늘 점심 질문을 실행하고 모델 답변의 텍스트를 출력
response = chain.invoke({"question": "오늘 점심 뭐 먹을까?"})
print(response.content)


2026년 10월 14일의 점심 메뉴를 추천해드릴게요!
1. **한식**: 제육볶음
2. **중식**: 짬뽕
3. **일식**: 연어덮밥
4. **양식**: 토마토 파스타

든든한 메뉴부터 가볍게 즐길 수 있는 메뉴까지 골라봤어요.
오늘은 입맛 당기는 메뉴로 맛있는 점심 드세요!


In [ ]:
# 수업자 자료
from datetime import datetime

from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 오늘 날짜를 반환하는 Runnable 생성
date_runnable = RunnableLambda(
    lambda _: {
        "today": datetime.now().strftime("%Y-%m-%d"),
        "question": "오늘 점심 뭐 먹을까?"
    }
)

# 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 음식 추천 어시스턴트입니다."),
    ("human", "오늘은 {today}. {question}")
])

# LLM 설정
llm = ChatOpenAI(
    model=default_model
)

# 문자열 파서
parser = StrOutputParser()

# 체인 구성 (LCEL 방식)
chain = date_runnable | prompt | llm | parser

# 실행
response = chain.invoke({})
print(response)

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from datetime import datetime
date_runnable = RunnableLambda(
    lambda query: {'today' : datetime.now().strftime('%Y%m%d'),
                    'query' : query })
prompt = ChatPromptTemplate.from_messages(
    [('system','오늘날짜 꼭 포함해줘'),
    ('user', '오늘 {today}입니다. {query}')]
)

chain = date_runnable | prompt | llm | StrOutputParser()
chain.invoke('오늘 점심 뭐 먹을까?')

'오늘은 **2026년 10월 14일**이에요. 점심으로는 따뜻한 **순두부찌개에 계란말이** 어떠세요? 든든하고 쌀쌀한 가을날에도 잘 어울려요.'

In [14]:
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=default_model)
parser = StrOutputParser()

# 장점을 설명하는 체인
pros_prompt = ChatPromptTemplate.from_template("{topic}의 장점을 한 문장으로 알려줘")
pros_chain = pros_prompt | llm | parser

# 단점을 설명하는 체인
cons_prompt = ChatPromptTemplate.from_template("{topic}의 단점을 한 문장으로 알려줘")
cons_chain = cons_prompt | llm | parser

# 두 체인을 병렬로 묶기
parallel_chain = RunnableParallel(pros=pros_chain, cons=cons_chain) 

result = parallel_chain.invoke({"topic": "전기차"})
print(result)
# 출력 예: {'pros': '전기차는 ...', 'cons': '전기차는 ...'}

{'pros': '전기차는 주행 중 배기가스를 배출하지 않고 에너지 효율이 높아 친환경적이며 유지비도 절감할 수 있습니다.', 'cons': '전기차는 충전 시간이 길고 충전소가 부족할 수 있으며, 초기 구매 비용이 높은 편입니다.'}


In [15]:
summary_prompt = ChatPromptTemplate.from_template(
    "장점: {pros}\n단점: {cons}\n\n위 내용을 종합해서 결론을 내려줘."
)

full_chain = parallel_chain | summary_prompt | llm | StrOutputParser()
print(full_chain.invoke("전기차를 사야할까?"))

전기차는 연료비와 배출가스를 줄이고 조용하고 부드러운 주행을 누릴 수 있다는 장점이 있지만, 높은 초기 구매비와 충전 시간·인프라, 배터리 교체 비용도 고려해야 합니다. 따라서 **충전 환경이 잘 갖춰져 있고 장기간 운행할 계획이라면 전기차가 좋은 선택이 될 수 있지만, 구매 전에는 총비용과 자신의 주행·충전 여건을 꼼꼼히 비교하는 것이 좋습니다.**


In [ ]:
from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough()

result = passthrough.invoke("안녕하세요")
print(result)
# 출력: 안녕하세요

In [16]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

chain = RunnableParallel(
    original=RunnablePassthrough(),
    upper=lambda x: x.upper()
)

result = chain.invoke("hello")
print(result)
# 출력: {'original': 'hello', 'upper': 'HELLO'}

{'original': 'hello', 'upper': 'HELLO'}


In [17]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template("{food}로 만들 수 있는 요리 하나만 추천해줘. 한 줄로.")
llm = ChatOpenAI(model="gpt-4o-mini")

chain = RunnablePassthrough() | prompt | llm

print(chain.invoke("계란").content)

계란 후라이를 얹은 비빔밥을 추천합니다!


In [18]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """두 숫자를 더합니다."""  # Docstring
    return a + b
    
@tool
def get_word_length(word: str) -> int:
    """단어나 문장의 정확한 글자 수(길이)를 계산할 때 사용합니다."""
    return len(word)    
           
print(add.name, "-", add.description)
print(add.args)
print(add.invoke({"a": 5, "b": 7}))    

add - 두 숫자를 더합니다.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
12


In [ ]:
tools = [add, get_word_length]
tools_list = {tool.name : tool for tool in tools} 

llm_with_tools = llm.bind_tools(tools)

query = "3과 5를 더해주세요"

# 1차 llm 호출 : llm_with_tools
res1 = llm_with_tools.invoke(query)

# print(tools_list.keys())

# tools_list = {'add': add, 'get_word_length': get_word_length}
# tools[tools_list['name']]

In [25]:
# tools로 add 함수를 실행을 해서 결과값을 반환받음
call = res1.tool_calls[0]

print(call['name'],call['args'])
# result = tools_list[call['name']].invoke(call['args'])
print(result)

add {'a': 3, 'b': 5}
{'original': 'hello', 'upper': 'HELLO'}


In [27]:
# 2차 llm 호출 : llm_with_tools
from langchain_core.messages import ToolMessage

messages = [
    ('user',query),
    res1,
    ToolMessage(tool_call_id = call['id'], content = str(result))
]
res2 = llm_with_tools.invoke(messages)
print(res2.text)

In [19]:
lst = [1,2,3,4]
print([i for i in lst])

[1, 2, 3, 4]


In [40]:
from langchain.agents import create_agent

agent = create_agent(
    model = llm,
    tools = [add,get_word_length],
    system_prompt = '문장의 글자 수는 get_word_length 도구로 계산하고, 덧셈은 add 도구로 계산하세요. 필요한 도구를 순서대로 사용한 뒤 결과를 답하세요.'

)
query = "'만나서 반갑습니다'의 글자 수를 계산한 뒤 5를 더해 결과를 알려주세요."

res = agent.invoke({'messages': [('user', query)]}, config={"recursion_limit": 10})
print(res['messages'][-1].content)

'만나서 반갑습니다'의 글자 수는 9이고, 여기에 5를 더하면 총 14입니다.


In [39]:
print(res['messages'][-1].text)

감사합니다! 어떤 도움이 필요하신가요?
